# 04 Generation

`src.generation` RAG 파이프라인을 구동하고 `src.evaluation` 답변 생성 품질을 hit / recall / bert score 기준 평가.



## Git Clone

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/songhahyun/finance-terms-rag-chatbot.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path('/content/finance-terms-rag-chatbot')

In [ ]:
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('cwd =', Path.cwd())

In [ ]:
# Python deps
!pip install -q -U pip
!pip install -q -r requirements.txt

# Ollama install + serve
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pandas tqdm

## Ollama setup

In [ ]:
import subprocess
import time

# Start Ollama server in the background
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give Ollama a moment to start up
time.sleep(5)
print("Ollama server started in the background.")

In [ ]:
# Ollama pull
!ollama pull qwen2.5:7b-instruct

os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_MODEL'] = 'qwen2.5:7b-instruct'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'

print('Ollama ready:', os.environ['OLLAMA_BASE_URL'])

## Fetch API Keys

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "OPENAI_API_KEY",
    "NCP_APIGW_API_KEY",
    "CLOVASTUDIO_API_KEY",
    "HF_TOKEN",
    "WANDB_API_KEY",
]

missing = []

for name in REQUIRED_SECRETS:
    value = userdata.get(name)
    if value:
        os.environ[name] = value
    else:
        missing.append(name)

if missing:
    raise RuntimeError(
        "Colab Secrets에 다음 값을 추가하고 Notebook access를 켜세요: "
        + ", ".join(missing)
    )

## Run experiment

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd

from src.common.config import get_settings
from src.evaluation.generation_pipeline import run_generation_experiment
from src.generation.language_validator import validate_answer_language
from src.generation.llm import OllamaGenerator

settings = get_settings()
VALIDATION_SMOKE_RESULT = validate_answer_language("ETF는 거래소에서 주식처럼 거래되는 금융상품입니다.")
if not VALIDATION_SMOKE_RESULT["is_valid"]:
    raise RuntimeError(f"Language validator smoke test failed: {VALIDATION_SMOKE_RESULT}")

EVAL_CSV_PATH = ROOT / "data" / "eval" / "testset" / "golden_testset_v2.csv"
CHUNK_JSON_PATH = settings.default_chunk_json_path
BM25_INDEX_PATH = CHUNK_JSON_PATH.with_name("bm25_index.pkl")
OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "generation_002_language_validator"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_WEAVE = True
WEAVE_PROJECT = "finance-terms-rag-evaluation"
WEAVE_EXPERIMENT_GROUP = "generation_002_language_validator"
WEAVE_LOG_CONTEXTS = True
WEAVE_LOG_PROMPT = True
WEAVE_PRINT_CALL_LINK = False
WEAVE_LOG_BATCH_SIZE = 10  # Set to 1 for row-by-row Weave logging, or 10 for small batches.
MAX_ROWS = None  # Set to a small integer, such as 5, for short CSV test runs.

EXPERIMENTS = [
    {
        "name": "hybrid_clova_bge-m3",
        "retrieval_mode": "hybrid",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "dense_clova_bge-m3",
        "retrieval_mode": "dense",
        "dense_provider": "clova",
        "dense_model_name": "bge-m3",
        "dense_collection_name": "docs_clova",
        "dense_persist_directory": str(settings.chroma_clova_dir),
    },
    {
        "name": "hybrid_openai_text-embedding-3-small",
        "retrieval_mode": "hybrid",
        "dense_provider": "openai",
        "dense_model_name": "text-embedding-3-small",
        "dense_collection_name": "docs_openai",
        "dense_persist_directory": str(settings.chroma_openai_dir),
    },
]

print("Eval CSV:", EVAL_CSV_PATH)
print("Chunk JSON:", CHUNK_JSON_PATH)
print("BM25 index PKL:", BM25_INDEX_PATH)
print("Output Dir:", OUTPUT_DIR)
print("Generation model:", settings.ollama_model)
print("Language validator:", VALIDATION_SMOKE_RESULT["reason"])
print("Max rows:", MAX_ROWS)
print("Use Weave:", USE_WEAVE)
print("Weave log batch size:", WEAVE_LOG_BATCH_SIZE)
print("Weave project:", WEAVE_PROJECT)
print("Weave experiment group:", WEAVE_EXPERIMENT_GROUP)


In [ ]:
warmup_generator = OllamaGenerator(
    model=settings.ollama_model,
    base_url=settings.ollama_base_url,
    timeout=settings.ollama_timeout,
    temperature=settings.ollama_temperature,
    top_p=settings.ollama_top_p,
    repeat_penalty=settings.ollama_repeat_penalty,
    keep_alive=settings.ollama_keep_alive,
)
warmup_answer = warmup_generator.generate("모델 warm-up 확인용 요청입니다. OK라고만 답하세요.")
print("Ollama warm-up complete:", warmup_answer[:80])

all_results = {}
summary_rows = []

for exp in EXPERIMENTS:
    output_path = OUTPUT_DIR / f"{exp['name']}.csv"
    result_df = run_generation_experiment(
        experiment_name=exp["name"],
        eval_csv_path=EVAL_CSV_PATH,
        chunk_json_path=CHUNK_JSON_PATH,
        output_csv_path=output_path,
        retrieval_mode=exp["retrieval_mode"],
        dense_provider=exp["dense_provider"],
        dense_model_name=exp["dense_model_name"],
        dense_collection_name=exp["dense_collection_name"],
        dense_persist_directory=exp["dense_persist_directory"],
        bm25_index_path=BM25_INDEX_PATH,
        ollama_model=settings.ollama_model,
        ollama_base_url=settings.ollama_base_url,
        ollama_timeout=settings.ollama_timeout,
        k=5,
        language="ko",
        use_weave=USE_WEAVE,
        weave_project=WEAVE_PROJECT,
        weave_experiment_group=WEAVE_EXPERIMENT_GROUP,
        weave_log_contexts=WEAVE_LOG_CONTEXTS,
        weave_log_prompt=WEAVE_LOG_PROMPT,
        weave_print_call_link=WEAVE_PRINT_CALL_LINK,
        weave_log_batch_size=WEAVE_LOG_BATCH_SIZE,
        max_rows=MAX_ROWS,
    )
    all_results[exp["name"]] = output_path

    summary_rows.append(
        {
            "experiment": exp["name"],
            "rows": len(result_df),
            "avg_recall": float(result_df["recall"].mean()) if len(result_df) else 0.0,
            "avg_mrr": float(result_df["mrr"].mean()) if len(result_df) else 0.0,
            "avg_bertscore_f1": float(result_df["bertscore_f1"].mean()) if len(result_df) else 0.0,
            "avg_stage_2_regeneration_count": float(result_df["stage_2_regeneration_count"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_bm25_latency_sec": float(result_df["stage_1_retrieval_bm25_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage1_1_retrieval_dense_latency_sec": float(result_df["stage1_1_retrieval_dense_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_fusion_latency_sec": float(result_df["stage_1_retrieval_fusion_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_1_retrieval_total_latency_sec": float(result_df["stage_1_retrieval_total_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_stage_2_generation_latency_sec": float(result_df["stage_2_generation_latency_sec"].mean()) if len(result_df) else 0.0,
            "avg_total_latency_sec": float(result_df["total_latency_sec"].mean()) if len(result_df) else 0.0,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_output_path = OUTPUT_DIR / "generation_experiment_summary.csv"
summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("Saved output files:")
for name, path in all_results.items():
    print(f"- {name}: {path}")
print(f"- summary: {summary_output_path}")

summary_df
